<a href="https://colab.research.google.com/github/shayanR10/FIV1/blob/main/to_ONNX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Required prerequisite
!pip install onnx -q onnxruntime -q onnxscript -q

In [ ]:
# This script allows you to convert your weights file into the ONNX format (so you can eventually use it, for example, inside a webapp).
# Make sure to upload your model file into the "Files" tab in Colab.

import torch
import torchvision.models as models
import onnx
import onnxruntime as onnxRT
from google.colab import files

weightPath = "resnet18DINOSAUR_best.safetensors"
stateDICT = torch.load(weightPath, map_location="cpu")

NUMgenera = stateDICT['fc.weight'].shape[0]
print(f"{NUMgenera} classes found.")

model = models.resnet18(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, NUMgenera)
model.load_state_dict(stateDICT)
model.eval()

print("loaded.")
print("starting export to ONNX")

dummy = torch.randn(1, 3, 224, 224, requires_grad=False)
ONNXpath = "resnet18DINOSAUR.onnx"

torch.onnx.export(
    model,
    dummy,
    ONNXpath,
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    }
)

print("export success.")

ONNXmodel = onnx.load(ONNXpath)
onnx.checker.check_model(ONNXmodel)

print("validated.")

files.download(ONNXpath)